Extract (verb, nsubj, obj, obj2, obl) vectors from pre-parsed Alpino XML files.

Output CSV format matches your old spaCy code:
sent_id, vector, sentence_text

Usage:
python extract_alpino_vectors.py
--input_dir /path/to/alpino_xml
--output_csv /path/to/vectors.csv
--save_every 10000
--target_vectors 1000000

In [1]:
import argparse
import csv
import os
from pathlib import Path
from typing import Optional, Tuple, List
import xml.etree.ElementTree as ET

In [7]:


# Alpino clause categories you’ll commonly see

CLAUSE_CATS = {
"smain", "ssub", "sv1", "inf", "ti", "rel", "whq", "oti",
# sometimes you’ll see these too; harmless to include
"cp", "du"
}

# Relations that often attach PP-like obliques to the verbal head

PP_OBL_RELS = {"mod", "pc", "ld", "me", "predm"}

def _attr_int(node: ET.Element, key: str, default: int = -1) -> int:
    v = node.get(key)
    if v is None:
        return default
    try:
        return int(v)
    except ValueError:
        return default

def _span(node: ET.Element) -> int:
    b = _attr_int(node, "begin", -1)
    e = _attr_int(node, "end", -1)
    if b < 0 or e < 0:
        return -1
    return e - b

def _is_verb(node: ET.Element) -> bool:
# Alpino uses pos="verb" on terminals; lcat/cat vary on non-terminals
    return node.get("pos") == "verb"

def _is_finite_verb(node: ET.Element) -> bool:
    # wvorm="pv" is finite verb form in Alpino terminals
    return _is_verb(node) and node.get("wvorm") == "pv"

def _iter_all_nodes(root: ET.Element):
    # Depth-first traversal of all <node> elements (including root if it is a <node>)
    stack = [root]
    while stack:
        n = stack.pop()
        yield n

        # IMPORTANT: this must be inside the while-loop
        children = list(n)
        # reverse so we keep document-ish order when using stack
        for c in reversed(children):
            if c.tag == "node":
                stack.append(c)


def _find_top_node(tree_root: ET.Element) -> Optional[ET.Element]:
    # Typically: <alpino_ds> -> <node cat="top" ...>
    for ch in tree_root:
        if ch.tag == "node" and ch.get("cat") == "top":
            return ch
    # fallback: search anywhere
    for n in tree_root.iter("node"):
        if n.get("cat") == "top":
            return n
    return None

def _find_sentence_text(tree_root: ET.Element) -> str:
    s = tree_root.find("sentence")
    if s is not None and s.text:
        return s.text.strip()
    return ""

def _head_terminal_lemma(node: ET.Element) -> Optional[str]:
    """
    Given any node (phrase/clause), try to find a good head terminal lemma:
    - Prefer a descendant with rel="hd" that is a terminal (has lemma/word).
    - If not found, prefer a terminal noun/pronoun/name.
    - Else any terminal with lemma.
    """
    # 1) preferred: terminal with rel=hd
    for n in _iter_all_nodes(node):
        if n.get("rel") == "hd" and n.get("lemma"):
            return n.get("lemma")


    # 2) noun/pronoun/name terminals as decent “content head”
    for n in _iter_all_nodes(node):
        pos = n.get("pos")
        if n.get("lemma") and pos in {"noun", "name", "pron"}:
            return n.get("lemma")

    # 3) any lemma-bearing terminal
    for n in _iter_all_nodes(node):
        if n.get("lemma"):
            return n.get("lemma")

    return None


def _find_best_clause(top: ET.Element) -> Optional[ET.Element]:
    """
    Choose the main clause for the sentence.
    Heuristic:
    - candidate if cat in CLAUSE_CATS and it contains a finite verbal head (rel=hd, wvorm=pv)
    - choose the one with the largest span (end-begin), tie-break by earliest begin
    """
    best = None
    best_span = -1
    best_begin = 10**9

    for n in top.iter("node"):
        cat = n.get("cat")
        if cat not in CLAUSE_CATS:
            continue

        # must have a finite head verb inside (usually rel=hd)
        has_finite = False
        for d in _iter_all_nodes(n):
            if d.get("rel") == "hd" and _is_finite_verb(d):
                has_finite = True
                break
        if not has_finite:
            continue

        sp = _span(n)
        b = _attr_int(n, "begin", 10**9)
        if sp > best_span or (sp == best_span and b < best_begin):
            best = n
            best_span = sp
            best_begin = b

    return best


def _find_main_verb_lemma(clause: ET.Element) -> Optional[str]:
    """
    Return lemma of the main finite verb (head) of the clause.
    """
    # Prefer the finite head verb directly under rel=hd if possible.
    for n in _iter_all_nodes(clause):
        if n.get("rel") == "hd" and _is_finite_verb(n) and n.get("lemma"):
            return n.get("lemma")

    # Fallback: any finite verb
    for n in _iter_all_nodes(clause):
        if _is_finite_verb(n) and n.get("lemma"):
            return n.get("lemma")

    # Last resort: any verb lemma
    for n in _iter_all_nodes(clause):
        if _is_verb(n) and n.get("lemma"):
            return n.get("lemma")

    return None

def _extract_su_obj_obj2(clause: ET.Element) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """
    Extract subject + up to two objects based on su/obj1/obj2 dependents.
    We take the head terminal lemma of the dependent phrase.
    """
    nsubj = obj = obj2 = None

    # Direct children are usually best, but sometimes they’re nested (e.g., clause wrappers).
    # We’ll scan the whole clause for nodes with these rels, preferring shallow ones.
    candidates: List[Tuple[int, ET.Element]] = []
    for n in _iter_all_nodes(clause):
        rel = n.get("rel")
        if rel in {"su", "obj1", "obj2"}:
            depth = 0
            p = n
            while p is not None and p is not clause:
                p = p.getparent() if hasattr(p, "getparent") else None  # xml.etree has no getparent
                depth += 1
                if depth > 1000:
                    break
            # xml.etree can't compute parent depth; approximate by span (smaller span often more specific)
            # We'll just use span as a proxy "priority"
            priority = _span(n)
            if priority < 0:
                priority = 10**9
            candidates.append((priority, n))

    # xml.etree limitation: no parent pointers; we’ll just sort by smaller span first
    candidates.sort(key=lambda x: x[0])

    for _, n in candidates:
        rel = n.get("rel")
        lemma = _head_terminal_lemma(n)
        if not lemma:
            continue
        if rel == "su" and nsubj is None:
            nsubj = lemma
        elif rel == "obj1":
            if obj is None:
                obj = lemma
            elif obj2 is None:
                obj2 = lemma
        elif rel == "obj2" and obj2 is None:
            obj2 = lemma

        if nsubj is not None and obj is not None and obj2 is not None:
            break

    return nsubj, obj, obj2

def _extract_obl(clause: ET.Element) -> Optional[str]:
    """
    Try to emulate spaCy's 'obl' by finding a PP-like dependent of the clause/verb
    and returning the lemma of the PP's object head (typically the noun inside NP).

    ```
    Heuristic:
      - Find nodes with cat="pp" and rel in PP_OBL_RELS
      - For each PP, prefer its obj1 NP head lemma; else fallback to PP head lemma
    """
    pps: List[ET.Element] = []
    for n in _iter_all_nodes(clause):
        if n.get("cat") == "pp" and n.get("rel") in PP_OBL_RELS:
            pps.append(n)

    # Choose the earliest PP in the sentence (by begin), similar to scanning children
    pps.sort(key=lambda n: _attr_int(n, "begin", 10**9))

    for pp in pps:
        # Prefer the object of the preposition (rel=obj1), then get its head lemma
        obj_np = None
        for ch in pp:
            if ch.tag == "node" and ch.get("rel") == "obj1":
                obj_np = ch
                break
        if obj_np is not None:
            lemma = _head_terminal_lemma(obj_np)
            if lemma:
                return lemma

        # Fallback: any decent head lemma from the PP
        lemma = _head_terminal_lemma(pp)
        if lemma and lemma not in {"van", "op", "in", "met", "naar", "voor", "door", "tijdens"}:
            return lemma

    return None

def extract_vector_from_alpino_xml(xml_path: Path,
                                   allow_copula=True,
                                   ) -> Tuple[Optional[Tuple[str, Optional[str], Optional[str], Optional[str], Optional[str]]], str]:
    """
    Returns: (vector_tuple or None, sentence_text)
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return None, ""

    sentence_text = _find_sentence_text(root)
    top = _find_top_node(root)
    if top is None:
        return None, sentence_text

    clause = _find_best_clause(top)
    if clause is None:
        return None, sentence_text

    verb = _find_main_verb_lemma(clause)
    if not verb:
        return None, sentence_text
    if not allow_copula:
        if verb == "zijn":
            return None, sentence_text

    # In your spaCy code you only keep sentences whose root POS is VERB.
    # Here we approximate by requiring we found a (finite) verb head.
    nsubj, obj, obj2 = _extract_su_obj_obj2(clause)
    obl = _extract_obl(clause)

    return (verb, nsubj, obj, obj2, obl), sentence_text

def iter_xml_files(input_dir: Path):
    for p in input_dir.rglob("*.xml"):
        yield p

def write_header_if_needed(output_csv: Path):
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    if not output_csv.exists():
        with output_csv.open("w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["sent_id", "vector", "sentence_text"])

In [3]:
from utils import DATA_DIR
from pathlib import Path
input_dir = Path("/home/nobackup/corpora/DutchGovernment/alpino")
output_csv = Path(DATA_DIR / "vectors" / "DutchGovernment_Test")
save_every = 100
target_vectors = 10000
sent_id = 1

write_header_if_needed(output_csv)

buffer: List[List[str]] = []
vec_count = 0

for xml_path in iter_xml_files(input_dir):
    vec, sent_text = extract_vector_from_alpino_xml(xml_path)
    # If you want to mimic spaCy’s “skip sentences with non-VERB root”,
    # we skip when vec is None.
    if vec is None:
        sent_id += 1
        continue

    buffer.append([sent_id, repr(vec), sent_text])
    vec_count += 1
    sent_id += 1

    if vec_count % save_every == 0:
        with output_csv.open("a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerows(buffer)
        buffer.clear()

    if target_vectors and vec_count >= target_vectors:
        break

if buffer:
    with output_csv.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerows(buffer)

print(f"Vectors written: {vec_count} (file: {output_csv})")

Vectors written: 5106 (file: /home/local/stefa/data/vectors/DutchGovernment_Test)


In [6]:
import random
from pathlib import Path
from typing import Dict, Iterator, List, Optional

def iter_xml_files_seeded_by_dir(
    input_dir: Path,
    *,
    seed: int,
    max_dirs: Optional[int] = None,
    shuffle_within_dir: bool = False,
) -> Iterator[Path]:
    """
    Reproducibly iterate XML files by randomly choosing directories until depleted.

    - seed: controls reproducible randomness
    - max_dirs: if set, only use that many directories (chosen reproducibly)
    - shuffle_within_dir: if True, randomize file order inside each directory too (seeded)
                          if False, files are yielded in sorted order within each directory
    """
    input_dir = Path(input_dir)
    if not input_dir.is_dir():
        raise NotADirectoryError(f"Not a directory: {input_dir}")

    rng = random.Random(seed)

    # 1) collect candidate directories (depth=1: your doc dirs)
    #    (sorted for determinism before applying RNG)
    doc_dirs = sorted([p for p in input_dir.iterdir() if p.is_dir()])
    # print(doc_dirs)

    # # Optional: if there are XMLs directly under input_dir, treat input_dir as a "doc dir"
    # if any(p.is_file() and p.suffix.lower() == ".xml" for p in input_dir.iterdir()):
    #     doc_dirs.insert(0, input_dir)

    if not doc_dirs:
        print("none found")
        return  # nothing to yield

    # 2) choose subset of directories reproducibly
    if max_dirs is not None and max_dirs < len(doc_dirs):
        print("sampling with seed", seed)
        # sample without replacement, reproducible
        doc_dirs = rng.sample(doc_dirs, k=max_dirs)

    # 3) build per-directory file lists
    #    (files are assumed to be directly inside the doc dir; adjust if needed)
    dir_to_files: Dict[Path, List[Path]] = {}
    print("starting file list creation")
    for d in doc_dirs:
        files = sorted([p for p in d.glob("*.xml") if p.is_file()])
        if not files:
            continue
        if shuffle_within_dir:
            rng.shuffle(files)  # seeded randomness
        dir_to_files[d] = files

    # If some chosen dirs had no XML, drop them
    print("sorting, checking for XML existence")
    active_dirs = sorted(dir_to_files.keys(), key=lambda p: p.as_posix())
    if not active_dirs:
        return

    # 4) repeatedly pick a directory at random until all are depleted
    while active_dirs:
        i = rng.randrange(len(active_dirs))
        d = active_dirs[i]
        files = dir_to_files[d]

        # pop next file from this directory
        nxt = files.pop(0)
        yield nxt

        # if directory depleted, remove it from active set
        if not files:
            active_dirs.pop(i)
            del dir_to_files[d]


In [8]:
input_dir = Path("/home/nobackup/corpora/Lassy/LASSY-met-POS/LASSY/Treebank")
output_csv = Path(DATA_DIR / "vectors" / "LASSY_test.csv")
save_every = 100
target_vectors = 10000
sent_id = 1

write_header_if_needed(output_csv)

buffer: List[List[str]] = []
vec_count = 0

for xml_path in iter_xml_files_seeded_by_dir(input_dir, seed=1):
    # print(xml_path)
    vec, sent_text = extract_vector_from_alpino_xml(xml_path, allow_copula=False)
    # print(vec, sent_text)
    # If you want to mimic spaCy’s “skip sentences with non-VERB root”,
    # we skip when vec is None.
    if vec is None:
        sent_id += 1
        continue

    buffer.append([sent_id, repr(vec), sent_text])
    vec_count += 1
    sent_id += 1

    if vec_count % save_every == 0:
        with output_csv.open("a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerows(buffer)
        buffer.clear()

    if target_vectors and vec_count >= target_vectors:
        break

if buffer:
    with output_csv.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerows(buffer)

print(f"Vectors written: {vec_count} (file: {output_csv})")

starting file list creation
sorting, checking for XML existence
Vectors written: 10000 (file: /home/local/stefa/data/vectors/LASSY_test)


In [14]:
# we print the first lines from the created csv
output_csv = Path(DATA_DIR / "vectors" / "LASSY_test.csv")
i = 0
with output_csv.open("r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if i > 10:
            break
        sent_id, vector, sentence = row[0], row[1], row[2]
        # the vectors have form "('verb', 'subject', 'object', 'rest', 'rest')"
        # we interpret the vector as a tuple of (verb, subject, object, *rest)
        vector = eval(vector)
        v, s, o = vector[0], vector[1], vector[2]
        print(f"{sent_id}, {sentence}\n\t{v}, {s}, {o}")
        i += 1

sent_id, sentence_text
	v, e, c
2, # Tijdens de 9e editie van GDMW Festival op 30 september en 1 oktober is de Rotterdamse Schouwburg gevuld met een unieke combinatie van grote literaire namen , aanstormend talent en literaire premières .
	zijn, Rotterdams, 30
3, Er zijn optredens van o.a. Kees van Kooten , Tim Krabbe , Tom Lanoye , Arthur Japin en Kristien Hemmerechts .
	zijn, optreden, Kees
4, Bekende namen wisselen af met new kids on the block als Ernest van der Kwast , Sanneke van Hassel , Richard Osinga , Liesbeth Mende en nationaal kampioen poetry slam Sander Koolwijk .
	wisselen, naam, kampioen
5, In totaal vinden er zo'n 45 optredens plaats op drie locaties in de tot festivalplek omgebouwde Rotterdamse Schouwburg .
	vinden, 45, totaal
8, Op 30 juli 2005 overleed te Amsterdam de fotograaf Philip Mechanicus ( * 1936 te Amsterdam ) .
	overlijden, fotograaf, Amsterdam
9, Mechanicus werd vooral bekend door zijn reeks schrijversportretten in NRC-Handelsblad in de jaren 1979 - 1981 , 